# RAGTruth EN vs PT-NLLB: Attention MIL vs Set Transformer

Este notebook é read-only e consolida a matriz fatorial: idioma de treino (EN original vs PT traduzido com NLLB) × agregador (Attention MIL vs Set Transformer). Ele separa efeito de tradução, efeito de agregador, interação descritiva, comportamento in-domain, transferência zero-shot, ranking, calibração, decisão binária e variabilidade entre seeds.

Nenhum modelo, tokenizer, checkpoint, CUDA, treinamento, inferência, tradução ou dataset é carregado/modificado.

## Research questions

1. A tradução EN → PT com NLLB altera a transferência para PublicHearingBR?
2. O Set Transformer supera Attention MIL dentro de cada idioma?
3. O efeito do agregador é maior na condição traduzida?
4. O ganho de ranking também aparece nas decisões binárias sob thresholds derivados somente da validation?

AUPRC é a métrica primária para ranking; AUROC e Brier são diagnósticos complementares. Para Brier, valores menores são melhores.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)

from ragtruth_transfer.factorial_paired_grouped_bootstrap import run_factorial_bootstrap

SEEDS = (0, 1, 2)
EXPECTED_ROWS, EXPECTED_POSITIVES, EXPECTED_HEARINGS = 4235, 501, 206
CONDITIONS = ("en_attention", "en_set", "pt_attention", "pt_set")
LABELS = {"en_attention": "EN Attention", "en_set": "EN Set", "pt_attention": "PT NLLB Attention", "pt_set": "PT NLLB Set"}
COLORS = {"en_attention": "#1f77b4", "en_set": "#ff7f0e", "pt_attention": "#2ca02c", "pt_set": "#d62728"}
REGISTRY = {
    "en_attention": {"language": "EN original", "translation": "No", "aggregator": "Attention MIL", "campaign": "runs/ragtruth_confirmatory/4e12933c51136624", "config": "configs/ragtruth_lora_attention_mil_confirmatory.yaml", "expected_architecture": "gated_attention"},
    "en_set": {"language": "EN original", "translation": "No", "aggregator": "Set Transformer", "campaign": "runs/ragtruth_confirmatory/28323e6cca11feb6", "config": "configs/ragtruth_lora_set_transformer_mil_confirmatory.yaml", "expected_architecture": "set_transformer"},
    "pt_attention": {"language": "PT NLLB", "translation": "NLLB", "aggregator": "Attention MIL", "campaign": "runs/ragtruth_pt_nllb_confirmatory/70bb1cce59b8c824", "config": "configs/ragtruth_pt_nllb_filtered_lora_attention_mil_confirmatory.yaml", "expected_architecture": "gated_attention"},
    "pt_set": {"language": "PT NLLB", "translation": "NLLB", "aggregator": "Set Transformer", "campaign": "runs/ragtruth_pt_nllb_set_transformer_confirmatory/29ec472694bb61a8", "config": "configs/ragtruth_pt_nllb_lora_set_transformer_confirmatory.yaml", "expected_architecture": "set_transformer"},
}

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks" / "03_pooling_ablation.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Repository root not found")

def load_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

ROOT = find_repo_root()
os.chdir(ROOT)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print(f"Repository: {ROOT}")

## Experiment registry and artifact integrity

Os paths abaixo são relativos ao repositório e centralizados em um único registry. Cada campanha é validada por manifest, configuração resolvida, seeds, histórico, predictions, thresholds e métricas agregadas.

In [ ]:
def validate_campaign(key, spec):
    campaign = ROOT / spec["campaign"]
    config = ROOT / spec["config"]
    assert campaign.is_dir(), f"Campaign missing: {campaign}"
    assert config.is_file(), f"Config missing: {config}"
    root_manifest = load_json(campaign / "manifest.json")
    resolved = load_json(campaign / "resolved_config.json")
    assert root_manifest.get("status") in {"externally_evaluated", "completed"}
    assert tuple(root_manifest.get("seeds", [])) == SEEDS, f"Seeds mismatch: {key}"
    assert resolved.get("architecture") == spec["expected_architecture"], f"Pooling mismatch: {key}"
    assert resolved.get("model_id") == "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
    assert resolved.get("encoder_mode") == "lora"
    data_audit = root_manifest.get("data_audit", {})
    dataset_signature = data_audit.get("metadata", {}).get("signature")
    split_signature = data_audit.get("split", {}).get("signature")
    assert dataset_signature and split_signature
    per_seed = []
    for seed in SEEDS:
        seed_dir = campaign / f"seed_{seed}"
        seed_manifest = load_json(seed_dir / "manifest.json")
        assert seed_manifest.get("status") in {"trained", "externally_evaluated"}
        assert seed_manifest.get("seed") == seed
        history_path = seed_dir / "history.csv"
        history = pd.read_csv(history_path)
        assert {"epoch", "validation_AUPRC"}.issubset(history.columns)
        assert (seed_dir / "validation_predictions.parquet").is_file()
        assert (seed_dir / "validation_metrics.json").is_file()
        assert (seed_dir / "thresholds.json").is_file()
        test_dir = seed_dir / "ragtruth_test"
        assert (test_dir / "predictions.parquet").is_file() and (test_dir / "metrics.json").is_file()
        target_manifests = sorted((seed_dir / "publichearing_zero_shot").glob("*/manifest.json"))
        assert len(target_manifests) == 1, f"Expected one target artifact: {key}/seed_{seed}"
        target_dir = target_manifests[0].parent
        target_manifest = load_json(target_manifests[0])
        assert target_manifest.get("status") in {"completed", "externally_evaluated"}
        target = pd.read_parquet(target_dir / "predictions.parquet")
        required = {"example_id", "hearing_id", "label", "probability"}
        assert required.issubset(target.columns)
        assert len(target) == EXPECTED_ROWS and target.example_id.is_unique
        assert target.hearing_id.nunique() == EXPECTED_HEARINGS and int(target.label.sum()) == EXPECTED_POSITIVES
        assert np.isfinite(target.probability.to_numpy(float)).all() and ((target.probability >= 0) & (target.probability <= 1)).all()
        per_seed.append({"seed": seed, "best_epoch": seed_manifest.get("best_epoch"), "best_validation_AUPRC": float(history.validation_AUPRC.max()), "history": history, "target": target, "target_dir": target_dir})
    aggregate = pd.read_csv(campaign / "aggregate" / "per_seed_metrics.csv")
    assert (campaign / "aggregate" / "aggregate_metrics.csv").is_file()
    return {"key": key, "campaign": campaign, "config": config, "manifest": root_manifest, "resolved": resolved, "dataset_signature": dataset_signature, "split_signature": split_signature, "per_seed": per_seed, "aggregate": aggregate}

campaigns = {key: validate_campaign(key, spec) for key, spec in REGISTRY.items()}
print("Artifact integrity passed for all four campaigns.")

## Experimental matrix and protocol caveats

As quatro condições usam mDeBERTa-v3-base multilingual NLI, LoRA e seeds 0/1/2. EN e PT têm populações RAGTruth comparáveis em tamanho, mas não a mesma assinatura: PT é uma view traduzida e possui split próprio. O target PublicHearingBR é o mesmo em todas as condições.

In [ ]:
control_rows = []
for key in CONDITIONS:
    item = campaigns[key]
    control_rows.append({"Condition": LABELS[key], "Training language": REGISTRY[key]["language"], "Translation": REGISTRY[key]["translation"], "Encoder": item["resolved"]["model_id"], "Aggregator": REGISTRY[key]["aggregator"], "Seeds": str(SEEDS), "Dataset population": f"{item['manifest']['data_audit']['rows']} RAGTruth rows; validation 4,517", "Target": f"PublicHearingBR: {EXPECTED_ROWS}/{EXPECTED_POSITIVES}", "Dataset signature": item["dataset_signature"], "Split signature": item["split_signature"]})
control_table = pd.DataFrame(control_rows)
display(control_table)

token_rows = []
for key in CONDITIONS:
    resolved = campaigns[key]["resolved"]
    token_rows.append({"Condition": LABELS[key], "max_length": resolved.get("max_length", 512), "truncation": resolved.get("truncation", "only_first"), "claim/pair token audit": "not present in frozen artifacts"})
display(pd.DataFrame(token_rows))
print("Caveat: EN uses the historical default only_first; PT explicitly uses longest_first. Token-count means/truncation incidence were not persisted in reproducible artifacts, so they are not reconstructed here.")

## Validation behavior

The next table and plot use `history.csv` only. They show best epoch and best validation AUPRC per seed, without selecting a new checkpoint or changing early stopping.

In [ ]:
validation_rows = []
for key in CONDITIONS:
    for row in campaigns[key]["per_seed"]:
        validation_rows.append({"Condition": LABELS[key], "condition": key, "seed": row["seed"], "best_epoch": row["best_epoch"], "best_validation_AUPRC": row["best_validation_AUPRC"]})
baseline_validation = pd.read_parquet(ROOT / "runs/ragtruth_off_the_shelf_threshold_transfer/3ceffc4a74b484fe/ragtruth_validation_predictions.parquet")
assert {"label", "hallucination_score"}.issubset(baseline_validation.columns)
baseline_validation_auprc = float(average_precision_score(baseline_validation["label"], baseline_validation["hallucination_score"]))
validation_rows.append({"Condition": "Off-the-shelf NLI baseline", "condition": "off_the_shelf", "seed": "deterministic", "best_epoch": "n/a", "best_validation_AUPRC": baseline_validation_auprc})
validation_summary = pd.DataFrame(validation_rows)
display(validation_summary)

fig, ax = plt.subplots(figsize=(14, 5))
for key in CONDITIONS:
    for seed in SEEDS:
        history = campaigns[key]["per_seed"][seed]["history"]
        ax.plot(history.epoch, history.validation_AUPRC, marker="o", alpha=.75, color=COLORS[key], label=LABELS[key] if seed == 0 else None)
ax.axhline(baseline_validation_auprc, color="#4d4d4d", linestyle="--", linewidth=2, label=f"Off-the-shelf NLI baseline ({baseline_validation_auprc:.3f})")
ax.set(xlabel="Epoch", ylabel="Validation AUPRC", title="Validation AUPRC by epoch")
ax.grid(axis="y", alpha=.2)
legend_handles, legend_labels = ax.get_legend_handles_labels()
fig.legend(legend_handles, legend_labels, loc="center left", bbox_to_anchor=(.76, .5), ncol=1, frameon=False)
fig.subplots_adjust(right=.74)
plt.show()

## RAGTruth test and PublicHearingBR: threshold-free metrics

These values are reconstructed from each campaign's `aggregate/per_seed_metrics.csv`. The displayed values are means across the three seeds; the tables also retain standard deviations for PublicHearingBR.

In [ ]:
metric_frames = []
for key in CONDITIONS:
    frame = campaigns[key]["aggregate"].copy()
    frame["condition"] = key
    frame["Condition"] = LABELS[key]
    metric_frames.append(frame)
metrics = pd.concat(metric_frames, ignore_index=True)
threshold_free = metrics[metrics.threshold_regime.eq("threshold_free")].copy()
summary = threshold_free.groupby(["dataset", "condition"])[["AUPRC", "AUROC", "Brier"]].agg(["mean", "std"])

def threshold_free_table(dataset):
    rows = []
    for key in CONDITIONS:
        rows.append({"Condition": LABELS[key], **{metric: summary.loc[(dataset, key), (metric, "mean")] for metric in ("AUPRC", "AUROC", "Brier")}})
    return pd.DataFrame(rows)

print("RAGTruth test")
display(threshold_free_table("ragtruth_test").round(4))
print("Table A — threshold-free RAGTruth test + PublicHearingBR")
table_a = []
for key in CONDITIONS:
    row = {"Training": REGISTRY[key]["language"], "Aggregator": REGISTRY[key]["aggregator"]}
    for dataset, prefix in (("ragtruth_test", "RAGTruth"), ("publichearing_zero_shot", "PH")):
        for metric in ("AUPRC", "AUROC", "Brier"):
            row[f"{prefix} {metric}"] = summary.loc[(dataset, key), (metric, "mean")]
    table_a.append(row)
display(pd.DataFrame(table_a).round(4))

def mean_metric(dataset, condition, metric):
    return float(summary.loc[(dataset, condition), (metric, "mean")])

effect_specs = {
    "EN: Set - Attention": ("en_set", "en_attention"),
    "PT: Set - Attention": ("pt_set", "pt_attention"),
    "Attention: PT - EN": ("pt_attention", "en_attention"),
    "Set: PT - EN": ("pt_set", "en_set"),
}
effect_rows = []
for contrast, (left, right) in effect_specs.items():
    effect_rows.append({"Contrast": contrast, **{metric: mean_metric("publichearing_zero_shot", left, metric) - mean_metric("publichearing_zero_shot", right, metric) for metric in ("AUPRC", "AUROC", "Brier")}})
effect_table = pd.DataFrame(effect_rows)
display(effect_table.round(5))

## Comparison of all systems: AUPRC and AUROC

The charts compare the four transfer-training conditions with the off-the-shelf NLI baseline and the supervised in-domain reference on PublicHearingBR in the threshold-free regime. Error bars show ±1 standard deviation for the four three-seed transfer conditions; the two reference systems are single frozen summaries and therefore have no seed error bar.

In [ ]:
bar_dataset = "publichearing_zero_shot"
off_the_shelf_metrics = load_json(ROOT / "runs/publichearing_off_the_shelf_max_entailment/54d9c623f8685c39/metrics.json")
in_domain_metrics = pd.read_csv(ROOT / "results/publichearing_lora_attention_mil/741ed3152c2175e7/outputs/overall_oof_metrics.csv").query("criterion == 'ranking'").iloc[0]
bar_rows = []
for key in CONDITIONS:
    item = summary.loc[(bar_dataset, key)]
    bar_rows.append({"condition": key, "Condition": LABELS[key], "Training": REGISTRY[key]["language"], "Aggregator": REGISTRY[key]["aggregator"], "AUPRC mean": item[("AUPRC", "mean")], "AUPRC std": item[("AUPRC", "std")], "AUROC mean": item[("AUROC", "mean")], "AUROC std": item[("AUROC", "std")]})
bar_rows.extend([
    {"condition": "off_the_shelf", "Condition": "Off-the-shelf NLI", "Training": "None", "Aggregator": "Frozen NLI head", "AUPRC mean": off_the_shelf_metrics["AUPRC"], "AUPRC std": np.nan, "AUROC mean": off_the_shelf_metrics["AUROC"], "AUROC std": np.nan},
    {"condition": "in_domain", "Condition": "In-domain supervised", "Training": "PublicHearingBR", "Aggregator": "Attention MIL", "AUPRC mean": float(in_domain_metrics["AUPRC"]), "AUPRC std": np.nan, "AUROC mean": float(in_domain_metrics["AUROC"]), "AUROC std": np.nan},
])
bar_summary = pd.DataFrame(bar_rows)
display(bar_summary[["Condition", "Training", "Aggregator", "AUPRC mean", "AUPRC std", "AUROC mean", "AUROC std"]].round(4))

bar_labels = {**LABELS, "off_the_shelf": "Off-the-shelf NLI", "in_domain": "In-domain supervised"}
bar_colors = {**COLORS, "off_the_shelf": "#4d4d4d", "in_domain": "#9467bd"}
bar_conditions = bar_summary["condition"].tolist()
x = np.arange(len(bar_conditions))
fig, axes = plt.subplots(1, 2, figsize=(15, 5.4), sharex=True)
for ax, metric in zip(axes, ("AUPRC", "AUROC")):
    means = bar_summary[f"{metric} mean"].to_numpy(float)
    errors = np.nan_to_num(bar_summary[f"{metric} std"].to_numpy(float), nan=0.0)
    bars = ax.bar(x, means, yerr=errors, capsize=4, color=[bar_colors[key] for key in bar_conditions], alpha=.9)
    ax.set_xticks(x, [bar_labels[key] for key in bar_conditions], rotation=22, ha="right")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} - PublicHearingBR", loc="left", weight="bold")
    ax.grid(axis="y", alpha=.2)
    for bar, value in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)
fig.suptitle("Comparison of all systems - mean ± SD where available", x=.06, ha="left", weight="bold")
plt.tight_layout()
plt.show()

## Factorial paired grouped bootstrap

The existing EN Set-vs-Attention artifact is audited separately. The factorial bootstrap below is needed because translation effects and the difference-in-differences interaction require the same sampled hearings across all four conditions. It uses 10,000 replicates, seed 20260815, `hearing_id` grouping, paired seeds, and the campaign statistic mean across paired seed effects.

In [ ]:
prediction_frames = {key: {row["seed"]: row["target"] for row in campaigns[key]["per_seed"]} for key in CONDITIONS}
en_existing_bootstrap = load_json(ROOT / "runs/publichearing_set_transformer_paired_bootstrap/3b55a2a78b11d58b/comparison_summary.json")
assert en_existing_bootstrap["delta_definition"] == "set_transformer - gated_attention"
assert en_existing_bootstrap["campaign"]["auprc"]["n_valid"] == 10000
print("Audited existing EN Set-vs-Attention bootstrap artifact:", en_existing_bootstrap["comparison"])
factorial = run_factorial_bootstrap(prediction_frames, seeds=SEEDS, n_replicates=10000, seed=20260815, metrics=("auprc",))

def bootstrap_contrast_table(result):
    rows = []
    for contrast in (*effect_specs.keys(), "interaction"):
        key = contrast if contrast == "interaction" else contrast.lower().replace(" - ", "_minus_").replace(": ", "_").replace(" ", "_")
        # Map display names to helper names explicitly.
        key = {"EN: Set - Attention": "en_set_minus_en_attention", "PT: Set - Attention": "pt_set_minus_pt_attention", "Attention: PT - EN": "pt_attention_minus_en_attention", "Set: PT - EN": "pt_set_minus_en_set", "interaction": "interaction"}[contrast]
        row = {"Contrast": contrast}
        for metric in ("auprc", "auroc", "brier"):
            row[f"{metric.upper()} Δ"] = factorial["observed_campaign"][key][metric]
            if metric in factorial["summaries"][key]:
                item = factorial["summaries"][key][metric]
                row[f"{metric.upper()} CI95%"] = f"[{item['ci_lower']:.5f}, {item['ci_upper']:.5f}]"
                row[f"{metric.upper()} P(Δ>0)"] = item["bootstrap_support_probability_delta_gt_zero"]
            else:
                row[f"{metric.upper()} CI95%"] = "not computed (AUPRC primary)"
                row[f"{metric.upper()} P(Δ>0)"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

display(bootstrap_contrast_table(factorial).round(5))
print("Factorial bootstrap completed in memory; no run artifact or dataset was modified.")

## Seed variability and ranking/calibration plots

Individual points are seeds; horizontal markers are the campaign means. The aggregator-effect plot uses the observed PublicHearing AUPRC deltas.

In [ ]:
ph_seed = threshold_free[threshold_free.dataset.eq("publichearing_zero_shot")].copy()
rt_seed = threshold_free[threshold_free.dataset.eq("ragtruth_test")].copy()

def seed_plot(frame, metric, title):
    fig, ax = plt.subplots(figsize=(8.8, 4.2))
    x = np.arange(len(CONDITIONS))
    for i, key in enumerate(CONDITIONS):
        values = frame.loc[frame.condition.eq(key), metric].to_numpy(float)
        ax.scatter(np.full(len(values), i), values, color=COLORS[key], s=38, alpha=.85)
        ax.plot([i-.18, i+.18], [values.mean(), values.mean()], color=COLORS[key], linewidth=3)
    ax.set_xticks(x, [LABELS[key] for key in CONDITIONS], rotation=15)
    ax.set_ylabel(metric); ax.set_title(title, loc="left", weight="bold"); ax.grid(axis="y", alpha=.2); plt.tight_layout(); plt.show()

seed_plot(ph_seed, "AUPRC", "PublicHearingBR AUPRC by seed")
seed_plot(rt_seed, "AUPRC", "RAGTruth test AUPRC by seed")

fig, ax = plt.subplots(figsize=(6.8, 4))
labels = ["EN", "PT NLLB"]
values = [factorial["observed_campaign"]["en_set_minus_en_attention"]["auprc"], factorial["observed_campaign"]["pt_set_minus_pt_attention"]["auprc"]]
bars = ax.bar(labels, values, color=["#8e5ea2", "#b279a2"], width=.55)
ax.axhline(0, color="black", linewidth=.8); ax.set_ylabel("ΔAUPRC = Set − Attention"); ax.set_title("Aggregator effect by training language", loc="left", weight="bold"); ax.grid(axis="y", alpha=.2)
for bar, value in zip(bars, values): ax.text(bar.get_x()+bar.get_width()/2, value, f"{value:+.4f}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, metric in zip(axes, ("auprc", "auroc", "brier")):
    vals = [factorial["observed_campaign"][name][metric] for name in ("en_set_minus_en_attention", "pt_set_minus_pt_attention")]
    ax.bar(["EN", "PT"], vals, color=["#8e5ea2", "#b279a2"]); ax.axhline(0, color="black", linewidth=.8); ax.set_title(metric.upper()); ax.grid(axis="y", alpha=.2)
axes[0].set_ylabel("Set − Attention"); fig.suptitle("Ranking versus calibration", x=.06, ha="left", weight="bold"); plt.tight_layout(); plt.show()

## Threshold-dependent behavior on PublicHearingBR

These tables and plots compare the four transfer-training conditions plus the off-the-shelf NLI baseline; the in-domain reference is intentionally excluded. They read frozen thresholded metrics from the aggregate artifacts. Thresholds were selected on the corresponding RAGTruth validation split and frozen on PublicHearingBR; no target re-optimization occurs here. The delta convention remains Set − Attention, so positive FPR delta is worse for Set.

In [ ]:
target_thresholded = metrics[(metrics.dataset.eq("publichearing_zero_shot")) & metrics.threshold_regime.isin(["best_f1", "fpr10"])].copy()
off_the_shelf_thresholded = load_json(ROOT / "runs/ragtruth_off_the_shelf_threshold_transfer/3ceffc4a74b484fe/publichearing_metrics.json")
threshold_metrics = ("Precision", "Recall", "F1", "MCC", "FPR")
def thresholded_table(regime):
    rows = []
    for key in CONDITIONS:
        subset = target_thresholded[(target_thresholded.condition == key) & (target_thresholded.threshold_regime == regime)]
        row = {"Condition": LABELS[key], "Training": REGISTRY[key]["language"], "Aggregator": REGISTRY[key]["aggregator"]}
        row.update({metric: float(subset[metric].mean()) for metric in threshold_metrics})
        rows.append(row)
    baseline = off_the_shelf_thresholded["off_the_shelf"][regime]
    rows.append({"Condition": "Off-the-shelf NLI", "Training": "None", "Aggregator": "Frozen NLI head", **{metric: baseline[metric] for metric in threshold_metrics}})
    return pd.DataFrame(rows)

print("Table B — PublicHearing best_f1")
display(thresholded_table("best_f1").round(4))
print("Table C — PublicHearing fpr10")
display(thresholded_table("fpr10").round(4))

threshold_plot_conditions = [*CONDITIONS, "off_the_shelf"]
threshold_plot_labels = {**LABELS, "off_the_shelf": "Off-the-shelf NLI"}
threshold_plot_colors = {**COLORS, "off_the_shelf": "#4d4d4d"}
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=False)
for row_index, regime in enumerate(("best_f1", "fpr10")):
    table = thresholded_table(regime)
    for col_index, metric in enumerate(("Precision", "Recall", "F1", "MCC")):
        ax = axes[row_index, col_index]
        means = table[metric].to_numpy(float)
        errors = np.array([target_thresholded[(target_thresholded.condition == key) & (target_thresholded.threshold_regime == regime)][metric].std() for key in CONDITIONS] + [0.0], dtype=float)
        bars = ax.bar(np.arange(len(threshold_plot_conditions)), means, yerr=np.nan_to_num(errors, nan=0.0), capsize=3, color=[threshold_plot_colors[key] for key in threshold_plot_conditions], alpha=.9)
        ax.set_xticks(np.arange(len(threshold_plot_conditions)), [threshold_plot_labels[key] for key in threshold_plot_conditions], rotation=35, ha="right")
        ax.set_title(f"{metric} - {regime}", loc="left", weight="bold")
        ax.grid(axis="y", alpha=.2)
        if col_index == 0:
            ax.set_ylabel("Metric value")
        for bar, value in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Threshold-dependent PublicHearingBR metrics", x=.06, ha="left", weight="bold")
plt.tight_layout()
plt.show()

threshold_effect_rows = []
for regime in ("best_f1", "fpr10"):
    for contrast, (left, right) in effect_specs.items():
        left_values = target_thresholded[(target_thresholded.condition == left) & (target_thresholded.threshold_regime == regime)]
        right_values = target_thresholded[(target_thresholded.condition == right) & (target_thresholded.threshold_regime == regime)]
        threshold_effect_rows.append({"Regime": regime, "Contrast": contrast, **{metric: float(left_values[metric].mean() - right_values[metric].mean()) for metric in ("Precision", "Recall", "F1", "MCC", "FPR")}})
display(pd.DataFrame(threshold_effect_rows).round(5))

pt_thresholded_bootstrap = load_json(ROOT / "runs/publichearing_pt_nllb_attention_vs_set_thresholded_bootstrap/e0c75270065fc471/bootstrap_summary.json")
display(pd.DataFrame([{"Regime": regime, "Metric": "F1", "Observed Δ": pt_thresholded_bootstrap[regime]["metrics"]["f1"]["observed_delta"], "CI95%": [pt_thresholded_bootstrap[regime]["metrics"]["f1"]["bootstrap"]["ci_lower"], pt_thresholded_bootstrap[regime]["metrics"]["f1"]["bootstrap"]["ci_upper"]]} for regime in ("best_f1", "fpr10")]).round(5))

## Ranking versus decision: conclusions

The interpretation is deliberately descriptive where the factorial interaction has no pre-registered hypothesis test. It does not claim that translation noise causes the Set Transformer effect or that Set compensates translation noise.

In [ ]:
en_translation = factorial["observed_campaign"]["pt_attention_minus_en_attention"]["auprc"]
set_translation = factorial["observed_campaign"]["pt_set_minus_en_set"]["auprc"]
en_set_effect = factorial["observed_campaign"]["en_set_minus_en_attention"]["auprc"]
pt_set_effect = factorial["observed_campaign"]["pt_set_minus_pt_attention"]["auprc"]
interaction = factorial["observed_campaign"]["interaction"]["auprc"]
interaction_summary = factorial["summaries"]["interaction"]["auprc"]

conclusions = [
    f"Q1 — NLLB translation improves PublicHearing transfer? NO on AUPRC: Attention Δ={en_translation:+.4f}; Set Δ={set_translation:+.4f}.",
    f"Q2 — Set improves EN? DESCRIPTIVELY YES (ΔAUPRC={en_set_effect:+.4f}); the existing EN campaign bootstrap is inconclusive for AUPRC.",
    f"Q3 — Set improves PT NLLB? YES: ΔAUPRC={pt_set_effect:+.4f}; the validated PT bootstrap is positive.",
    f"Q4 — Is Set's advantage larger in PT? DESCRIPTIVELY YES: interaction={interaction:+.4f}; factorial bootstrap CI=[{interaction_summary['ci_lower']:+.4f}, {interaction_summary['ci_upper']:+.4f}] and P(Δ>0)={interaction_summary['bootstrap_support_probability_delta_gt_zero']:.4f}.",
    "Q5 — Does ranking gain translate to decisions? EN: F1 improves descriptively; PT: F1 and Recall improve robustly under both frozen regimes.",
    "Q6 — Main trade-off: Set generally gains Recall/F1, while Precision and FPR can worsen; PT also shows a calibration trade-off in the broader comparison.",
    "Ranking, calibration and thresholded decisions must remain separate claims; no causal explanation is established here.",
]
for conclusion in conclusions:
    print(f"- {conclusion}")

assert set(campaigns) == set(CONDITIONS)
assert factorial["protocol"]["n_replicates"] == 10000 and factorial["protocol"]["seed"] == 20260815
assert all(factorial["summaries"][contrast][metric]["n_valid"] == 10000 for contrast in (*factorial["summaries"],) for metric in factorial["protocol"]["metrics"])
print("All notebook sanity checks passed. No model/GPU/inference path was used.")